**01 - Loading and cleaning the fact checks**

Source: Data Commons ClaimReview feed (Google, CC BY, updated daily, no api key needed)

Goal: keep only Indian fact checks in English + Hindi, clean publisher names, ratings and dates, and save to SQLite so the RAG part (embeddings + Chroma) can use it

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import re
import json
import sqlite3
import unicodedata
import requests
from urllib.parse import urlparse

Loading The **Data**

In [3]:
FEED_URL="https://storage.googleapis.com/datacommons-feeds/claimreview/latest/data.json"
RAW_PATH="../data/raw/claimreview_feed.json"

#downloading only if we dont have it already(file is ~190MB)
if not os.path.exists(RAW_PATH):
  os.makedirs("../data/raw",exist_ok=True)
  r=requests.get(FEED_URL,timeout=120)
  with open(RAW_PATH,"wb") as f:
    f.write(r.content)
print(os.path.getsize(RAW_PATH)/1e6,"MB")

201.039815 MB


In [4]:
with open(RAW_PATH,encoding="utf-8") as f:
  feed=json.load(f)

In [5]:
feed.keys()

dict_keys(['@context', '@type', 'dataFeedElement', 'dateModified', 'name'])

In [6]:
elements=feed["dataFeedElement"]
len(elements)

94134

In [7]:
#one fact check looks like this(item is a list and has the actual ClaimReview)
elements[0]["item"][0]

{'@context': 'http://schema.org',
 '@type': 'ClaimReview',
 'author': {'@type': 'Organization',
  'name': 'فکت\u200cنامه',
  'url': 'https://factnameh.com/'},
 'claimReviewed': 'ویدیوی واکنش عصبانی ترامپ به نتایج انتخابات آمریکا',
 'datePublished': '2026-11-06',
 'itemReviewed': {'@type': 'Claim',
  'author': {'@type': 'Person', 'name': 'شبکه\u200cهای اجتماعی'}},
 'reviewRating': {'@type': 'Rating', 'alternateName': 'شاخ\u200cدار'},
 'sdPublisher': {'@type': 'Organization',
  'name': 'Google Fact Check Tools',
  'url': 'https://g.co/factchecktools'},
 'url': 'https://factnameh.com/fa/fact-checks/2026-11-06-fake-video-trump-election-results'}

In [8]:
#flattening the nested json into rows(only the fields we need)
rows=[]
for el in elements:
  for item in el.get("item") or []:
    author=item.get("author") or {}
    if isinstance(author,list):
      author=author[0]
    rating=item.get("reviewRating") or {}
    rows.append({
        "claim":item.get("claimReviewed"),
        "rating_raw":rating.get("alternateName"),
        "publisher_raw":author.get("name"),
        "url":item.get("url"),
        "date":item.get("datePublished")
    })
df=pd.DataFrame(rows)

In [9]:
df.shape

(100479, 5)

In [10]:
df.head()

,claim,rating_raw,publisher_raw,url,date
0,ویدیوی واکنش عصبانی ترامپ به نتایج انتخابات آم...,شاخ‌دار,فکت‌نامه,https://factnameh.com/fa/fact-checks/2026-11-0...,2026-11-06
1,سرطان پانکراس پس از ایمونوتراپی جدید mRNA به م...,گمراه‌کننده,فکت‌نامه,https://factnameh.com/fa/fact-checks/2026-11-0...,2026-11-06
2,அஸ்வெசும பயனாளிகள் பட்டியலிலிருந்து சுமார் 14 ...,சூழல் விடுபட்டது,Fact Crescendo Sri Lanka,https://srilanka.factcrescendo.com/tamil/fact-...,2026-11-05
3,"தைஸ், அல் ஜவ்ஃப், லஹ்ஜ் ஆகியவை ஹூதி படைகளின் ம...",சூழல் விடுபட்டது,Fact Crescendo Sri Lanka,https://srilanka.factcrescendo.com/tamil/have-...,2026-11-05
4,2026 ඔක්තෝබර් 28 හෝ ඉන් පසු ටෙල් අවිව්හිදී නෙත...,අසත්‍යයි,Fact Crescendo Sri Lanka,https://srilanka.factcrescendo.com/2026/11/06/...,2026-11-06


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100479 entries, 0 to 100478
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   claim          99770 non-null   str  
 1   rating_raw     99708 non-null   str  
 2   publisher_raw  98768 non-null   str  
 3   url            100479 non-null  str  
 4   date           89638 non-null   str  
dtypes: str(5)
memory usage: 3.8 MB


In [12]:
df.isnull().sum()

claim              709
rating_raw         771
publisher_raw     1711
url                  0
date             10841
dtype: int64

**Publisher** cleaning

In [13]:
#the names are typed by hand by each fact checker so the same publisher comes in many spellings
df["publisher_raw"].str.lower().str.contains("factly").sum()

np.int64(12449)

In [14]:
df[df["publisher_raw"].str.lower().str.contains("factly",na=False)]["publisher_raw"].value_counts()

publisher_raw
FACTLY                                                                                                                                 12373
FACTLY                                                                                                                                    53
Factly                                                                                                                                     9
FACTLY                                                                                                                                     3
factly                                                                                                                                     2
FACTLY                                                                                                                                     1
FACTLYhttps://www.facebook.com/dishant.patel.75248795/posts/pfbid0DsBdnjv6Aj9tED4hZEVDMf77hHYXFLVFzgmAFJwFC9PWfE4cHiuRp5sZWw9W67Ffl        1

In [15]:
#so instead of the name im using the website domain from the url(it cant be misspelled)
df["domain"]=df["url"].fillna("").apply(lambda u: urlparse(u).netloc.lower().replace("www.",""))
df["domain"].value_counts().head(20)

domain
factly.in                     12474
politifact.com                 8165
dpa-factchecking.com           5534
boatos.org                     4268
youturn.in                     4038
lupa.uol.com.br                3369
en.youturn.in                  3118
verafiles.org                  3109
factcrescendo.com              3025
factnameh.com                  2945
pesacheck.org                  2883
logicallyfacts.com             2828
srilanka.factcrescendo.com     2680
factcheck.org                  2458
tamil.factcrescendo.com        2454
thequint.com                   2080
newtral.es                     1609
univision.com                  1430
vishvasnews.com                1321
fastcheck.cl                   1275
Name: count, dtype: int64

In [16]:
#only indian english/hindi editions(fact crescendo srilanka,cambodia,thailand etc are left out on purpose)
PUBLISHERS={
    "factly.in":"FACTLY",
    "factcrescendo.com":"Fact Crescendo",
    "english.factcrescendo.com":"Fact Crescendo",
    "logicallyfacts.com":"Logically Facts",
    "logically.ai":"Logically Facts",
    "logically.in":"Logically Facts",
    "thequint.com":"The Quint",
    "hindi.thequint.com":"The Quint",
    "fit.thequint.com":"The Quint",
    "vishvasnews.com":"Vishvas News",
    "thip.media":"THIP Media",
    "newsmeter.in":"NewsMeter",
    "newschecker.in":"Newschecker",
    "eastmojo.com":"EastMojo",
    "dfrac.org":"DFRAC",
    "theprobe.in":"The Probe",
    "newsmobile.in":"NewsMobile",
    "boomlive.in":"BOOM",
    "hindi.boomlive.in":"BOOM",
    "indiatoday.in":"India Today"
}
df["publisher"]=df["domain"].map(PUBLISHERS)

In [17]:
df["publisher"].isnull().sum()

np.int64(76643)

In [18]:
#dropping everything thats not an indian publisher
df=df.dropna(subset=["publisher"])
df.shape

(23836, 7)

In [19]:
df["publisher"].value_counts()

publisher
FACTLY             12474
Fact Crescendo      3236
Logically Facts     2926
The Quint           2437
Vishvas News        1321
THIP Media          1100
NewsMeter            153
Newschecker          111
EastMojo              33
DFRAC                 24
The Probe              8
BOOM                   6
NewsMobile             6
India Today            1
Name: count, dtype: int64

**Language** (english + hindi only for version 1)

In [20]:
#the feed has no language field so detecting it from the script of the claim text
#devanagari unicode block is 0900-097F,bengali to sinhala scripts are 0980-0DFF,urdu/arabic is 0600-06FF
def detect_lang(text):
  if re.search(r"[\u0900-\u097F]",text):
    return "hi"
  if re.search(r"[\u0980-\u0DFF\u0600-\u06FF]",text):
    return "other"
  return "en"

In [21]:
df["claim"]=df["claim"].fillna("").str.strip()
df["lang"]=df["claim"].apply(detect_lang)
df["lang"].value_counts()

lang
en       12636
other     6393
hi        4807
Name: count, dtype: int64

In [22]:
#telugu factly articles are big part of "other",leaving them for later(To do)
df=df[df["lang"].isin(["en","hi"])]
df.shape

(17443, 8)

**Rating** cleaning

In [23]:
df["rating_raw"].nunique()

413

In [24]:
df["rating_raw"].str.lower().value_counts().head(30)

rating_raw
false                                                      11267
misleading                                                  2178
false                                                       1383
mostly false                                                 425
true                                                         243
fact crescendo rating false                                  237
মিছা                                                         231
fake                                                         191
missing context                                              153
                                                 false        97
missing context                                               77
partly false                                                  76
misleading                                                    63
half true                                                     60
বিভ্ৰান্তিকৰ                                                  55
unverified    

In [25]:
#grouping 100s of spellings into 5 labels
#order matters:"mostly false" before "false",and "half true" before "true"
#logically facts assamese desk writes english claims with assamese labels(মিছা=false,ভুৱা=fake,বিভ্ৰান্তিকৰ=misleading)
SATIRE_WORDS=["satire","व्यंग्य"]
MISLEADING_WORDS=["misleading","missing context","partly false","half true","mixture","partly true","out of context","भ्रामक","বিভ্ৰান্তিকৰ"]
FALSE_WORDS=["false","fake","altered","fabricated","edited","untrue","असत्य","फ़ेक","गलत","झूठ","মিছা","ভুৱা"]
TRUE_WORDS=["true","correct","real news","सच","सही"]

#hindi "फ़" can be typed as 1 character or as फ + a dot(nukta),looks same but unicode is different
#NFC normalisation turns both into the same form so the match works
def clean_text(s):
  return unicodedata.normalize("NFC",str(s)).lower().strip()

SATIRE_WORDS=[clean_text(w) for w in SATIRE_WORDS]
MISLEADING_WORDS=[clean_text(w) for w in MISLEADING_WORDS]
FALSE_WORDS=[clean_text(w) for w in FALSE_WORDS]
TRUE_WORDS=[clean_text(w) for w in TRUE_WORDS]

def map_verdict(rating):
  r=clean_text(rating)
  if any(w in r for w in SATIRE_WORDS):
    return "SATIRE"
  if "mostly false" in r:
    return "FALSE"
  if any(w in r for w in MISLEADING_WORDS):
    return "MISLEADING"
  if any(w in r for w in FALSE_WORDS):
    return "FALSE"
  if any(w in r for w in TRUE_WORDS):
    return "TRUE"
  return "OTHER"

In [26]:
#testing on some tricky ones before applying
for r in ["Fact Crescendo Rating: False","Mostly False","Half True","Missing Context","असत्य","false. वायरल तस्वीर 12 साल पुरानी है","Satire","Explainer","Untrue","মিছা","फ़ेक","\u092b\u093c\u0947\u0915","Real News"]:
  print(r,"-->",map_verdict(r))

Fact Crescendo Rating: False --> FALSE
Mostly False --> FALSE
Half True --> MISLEADING
Missing Context --> MISLEADING
असत्य --> FALSE
false. वायरल तस्वीर 12 साल पुरानी है --> FALSE
Satire --> SATIRE
Explainer --> OTHER
Untrue --> FALSE
মিছা --> FALSE
फ़ेक --> FALSE
फ़ेक --> FALSE
Real News --> TRUE


In [27]:
df["verdict"]=df["rating_raw"].apply(map_verdict)
df["verdict"].value_counts()

verdict
FALSE         14239
MISLEADING     2856
TRUE            254
OTHER            90
SATIRE            4
Name: count, dtype: int64

In [28]:
#checking what went into OTHER so nothing important is lost there
#str[:40] cuts the long ones so the output is readable
df[df["verdict"]=="OTHER"]["rating_raw"].str.strip().str.lower().str[:40].value_counts().head(20)

rating_raw
unverified                                  52
fasle                                        2
x                                            2
logically facts                              2
1                                            2
unverifiable                                 2
the picture does not show indian migrant     1
dfrac टीम ने वायरल दावे की पड़ताल के लिए     1
we did not find any such official news r     1
in reality, bsf has rejected these repor     1
open letter                                  1
staire                                       1
explainer                                    1
miissing context                             1
in reality, tejas fighter jet made an em     1
flase. वायरल तस्वीर में एडिटिंग करते हुए     1
missleading                                  1
social  a video of a comedy scene from a     1
the viral video is old (2022). this has      1
dfrac found that the viral video of pm m     1
Name: count, dtype: int64

**Date** cleaning

In [29]:
df["date"]=pd.to_datetime(df["date"],errors="coerce")
df["date"].describe()

count                         17327
mean     2022-07-19 01:42:48.223004
min             2001-03-03 00:00:00
25%             2020-09-02 00:00:00
50%             2022-09-21 00:00:00
75%             2024-03-12 00:00:00
max             2923-01-02 00:00:00
Name: date, dtype: object

In [30]:
#some dates are junk(1970,2923,missing),keeping 2015 to today
valid_dates=(df["date"]>="2015-01-01") & (df["date"]<=pd.Timestamp.today())
print("bad dates:",(~valid_dates).sum())
df=df[valid_dates]

bad dates: 122


**Duplicates** and empty claims

In [31]:
print("empty claims:",(df["claim"]=="").sum())
print("duplicate urls:",df["url"].duplicated().sum())

empty claims: 19
duplicate urls: 349


In [32]:
df=df[df["claim"]!=""]
df=df.drop_duplicates(subset=["url"])
df.shape

(16953, 9)

Final **Check**

In [33]:
#keeping only the columns the app needs(no full article text,only claim + rating + link)
df=df[["claim","verdict","rating_raw","publisher","url","date","lang"]].reset_index(drop=True)
df.head()

,claim,verdict,rating_raw,publisher,url,date,lang
0,Video of Muslim MPs performing Namaz together ...,FALSE,FALSE,FACTLY,https://factly.in/video-from-jordans-parliamen...,2026-11-05,en
1,The video shows people chasing Union Home Mini...,FALSE,FALSE,FACTLY,https://factly.in/this-viral-video-does-not-sh...,2026-11-06,en
2,The video shows External Affairs Minister S Ja...,FALSE,FALSE,FACTLY,https://factly.in/this-viral-video-of-minister...,2026-11-05,en
3,"Visuals of police parading two Muslim men, ide...",MISLEADING,MISLEADING,FACTLY,https://factly.in/video-of-youths-paraded-for-...,2026-11-04,en
4,The video shows a real incident of a girl bein...,FALSE,FALSE,FACTLY,https://factly.in/this-viral-video-of-a-girl-b...,2026-11-04,en


In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16953 entries, 0 to 16952
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   claim       16953 non-null  str           
 1   verdict     16953 non-null  str           
 2   rating_raw  16950 non-null  str           
 3   publisher   16953 non-null  str           
 4   url         16953 non-null  str           
 5   date        16953 non-null  datetime64[us]
 6   lang        16953 non-null  str           
dtypes: datetime64[us](1), str(6)
memory usage: 927.2 KB


In [35]:
pd.crosstab(df["publisher"],df["verdict"])

verdict,FALSE,MISLEADING,OTHER,SATIRE,TRUE
publisher,,,,,
BOOM,3,3,0,0,0
DFRAC,5,9,8,0,1
EastMojo,26,7,0,0,0
FACTLY,5111,1581,8,0,69
Fact Crescendo,2718,480,8,1,10
Logically Facts,1943,445,15,0,0
NewsMeter,147,3,1,0,0
NewsMobile,5,0,1,0,0
Newschecker,33,26,1,2,1


In [36]:
pd.crosstab(df["publisher"],df["lang"])

lang,en,hi
publisher,,
BOOM,5,1
DFRAC,21,2
EastMojo,33,0
FACTLY,6768,1
Fact Crescendo,1151,2066
Logically Facts,1842,561
NewsMeter,151,0
NewsMobile,6,0
Newschecker,22,41


Saving to **SQLite**

In [37]:
DB_PATH="../data/factchecks.db"
conn=sqlite3.connect(DB_PATH)
df.to_sql("factchecks",conn,if_exists="replace",index=False)

16953

In [38]:
#reading back with sql to verify
pd.read_sql("SELECT verdict,COUNT(*) AS n FROM factchecks GROUP BY verdict ORDER BY n DESC",conn)

,verdict,n
0,FALSE,14053
1,MISLEADING,2767
2,TRUE,86
3,OTHER,43
4,SATIRE,4


In [39]:
pd.read_sql("SELECT publisher,MIN(date) AS first,MAX(date) AS last,COUNT(*) AS n FROM factchecks GROUP BY publisher ORDER BY n DESC",conn)

,publisher,first,last,n
0,FACTLY,2015-06-15 00:00:00,2026-11-06 00:00:00,6769
1,Fact Crescendo,2018-12-24 00:00:00,2026-05-30 00:00:00,3217
2,The Quint,2018-07-03 00:00:00,2024-05-15 00:00:00,2434
3,Logically Facts,2019-12-27 00:00:00,2025-05-16 00:00:00,2403
4,THIP Media,2020-04-30 00:00:00,2025-06-13 00:00:00,1038
5,Vishvas News,2018-12-11 00:00:00,2025-03-09 00:00:00,802
6,NewsMeter,2019-07-14 00:00:00,2024-03-14 00:00:00,151
7,Newschecker,2019-03-01 00:00:00,2024-05-02 00:00:00,63
8,EastMojo,2024-03-29 00:00:00,2025-01-23 00:00:00,33
9,DFRAC,2023-09-15 00:00:00,2025-02-06 00:00:00,23


In [40]:
conn.close()

**Insights**

1.83% of the fact checks are FALSE(14,053) and only 86 are TRUE.fact checkers mostly write about false claims,so the data is very imbalanced

---

2.MISLEADING is the second biggest group(2,767).a lot of fake news is a real photo or video with a wrong story,not fully made up

---

3.FACTLY alone is 40% of the data.The Quint stops in 2024 and Logically Facts in 2025,so the feed alone gets old.this is why the app also asks google's live api

---

4.OTHER went down from 380 to 43 after adding the assamese words and fixing the unicode(NFC).ratings are written in many languages and spellings

To do:

1.Add telugu and other indian languages(check if the embedding model handles them first)

---

2.Check 30 rows by hand against the source url(is the stored verdict same as the article?)

---

3.Look at the OTHER bucket again,some might belong to a real label

---

4.Punjabi claims are tagged as hindi(gurmukhi uses the same । as devanagari)